In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [2]:
groq = client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

ollama = OpenAI(
    api_key="http://127.0.0.1:11434/v1",
    base_url="http://127.0.0.1:11434/v1",
)

models = {
    "openai/gpt-oss-20b": groq,
    "llama-3.3-70b-versatile": groq,
    "qwen/qwen3-32b": groq,
    "moonshotai/kimi-k2-instruct-0905": groq,
    "gemma3:270m": ollama,
    "llama3.2:1b": ollama,
    "phi4-mini:latest": ollama
}

In [3]:
sys_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
Your response will be written to a file called main.cpp and then compiled and executed.
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""





python = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [4]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)


compile_command = ["clang++", "-std=c++17", "-Ofast", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]


def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)


def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [5]:
def port_code(model):
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

    client = models[model]

    res = client.chat.completions.create(model=model, messages=messages)
    code = res.choices[0].message.content
    code = code.replace("```cpp", "").replace("```", "")
    
    return code


In [6]:
code = port_code("moonshotai/kimi-k2-instruct-0905")
write_output(code)

In [7]:
print("Python result")
run_python(python)

Python result
Result: 3.141592656089
Execution Time: 56.671428 seconds


In [8]:
print("C++ result")
compile_and_run()

C++ result
Result: 3.141592656089
Execution Time: 0.384248 seconds

Result: 3.141592656089
Execution Time: 0.405410 seconds

Result: 3.141592656089
Execution Time: 0.424734 seconds

